## بخش 0: نصب وابستگی‌ها (فقط یکبار اجرا کن)

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## بخش 1: Import کتابخانه‌ها

In [2]:
import os
import re
import json
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel
import torch

print("✅ کتابخانه‌ها بارگذاری شدند")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ کتابخانه‌ها بارگذاری شدند


In [3]:
import torch
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"

def load_base_model():
    print("\n" + "="*80)
    print("📦 بارگذاری مدل پایه")
    print("="*80)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        dtype=None,            # auto
        load_in_4bit=True
    )

    FastLanguageModel.for_inference(model)
    print("✅ مدل با موفقیت لود شد!")

    return model, tokenizer


def test_model(model, tokenizer):
    """تست سریع مدل پایه"""
    print("\n" + "="*80)
    print("🧪 تست مدل پایه (بدون Fine-tune)")
    print("="*80)

    test_prompts = [
        # 1️⃣ پایه‌ای (MBPP)
        "Write a Python function that returns the sum of all even numbers in a list.",

        # 2️⃣ رشته‌ای
        "Write a Python function that checks whether a string is a palindrome, ignoring spaces and case.",

        # 3️⃣ منطقی کوتاه
        "Write a Python function that returns the first non-repeating character in a string.",

        # 4️⃣ الگوریتمی سبک
        "Write a Python function that merges two sorted lists into one sorted list.",

        # 5️⃣ عددی / لبه‌ای
        "Write a Python function that returns True if a number is a prime number, otherwise False."
    ]

    for i, problem in enumerate(test_prompts, 1):
        print(f"\n{'='*80}")
        print(f"🧪 تست {i}")
        print(f"{'='*80}")

        prompt = f"""<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
{problem}<|im_end|>
<|im_start|>assistant
"""

        print("📝 پرامپت:")
        print(prompt)
        print("\n🤖 پاسخ مدل:")
        print("-" * 80)

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        print(response[len(prompt):])
        print("-" * 80)

    print("\n✅ تست مدل پایه کامل شد!")


# اجرای کامل
model, tokenizer = load_base_model()
test_model(model, tokenizer)

print("\n" + "="*80)
print("🎉 تست مدل پایه با موفقیت انجام شد!")
print("="*80)



📦 بارگذاری مدل پایه
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ مدل با موفقیت لود شد!

🧪 تست مدل پایه (بدون Fine-tune)

🧪 تست 1
📝 پرامپت:
<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Write a Python function that returns the sum of all even numbers in a list.<|im_end|>
<|im_start|>assistant


🤖 پاسخ مدل:
--------------------------------------------------------------------------------
Certainly! Below is a Python function that calculates the sum of all even numbers in a given list:

```python
def sum_of_even_numbers(numbers):
    """
    Return the sum of all even numbers in the provided list.

    :param numbers: List of integers
    :return: Sum of even numbers
    """
    return sum(num for num in numbers if num % 2 == 0)

# Example usage:
numbers = [1, 2, 3, 4, 5, 6]
print(sum_of_even_numbers(numbers))  # Output: 12
```

### Explanation:
- The function `sum_of_even_numbers` takes a list of integers as its parameter.
- It uses a generator expression within the `sum()

## بخش 2: تنظیمات پروژه

In [3]:
import torch

class Config:
    """تنظیمات بهینه‌شده برای GPU 14GB - اصلاح‌شده"""

    # Paths
    DRIVE_MOUNT = "/content/drive"
    OUTPUT_DIR = "./qwen-coder-checkpoints"
    FINAL_MODEL_DIR = "./qwen-coder-final"
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/qwen-coder-humaneval"

    # Model Settings
    MODEL_NAME = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"
    MAX_SEQ_LENGTH = 2048
    DTYPE = None  # Auto detection
    LOAD_IN_4BIT = True

    # LoRA Configuration - اصلاح‌شده ✅
    LORA_R = 16
    LORA_ALPHA = 32  # ✅ تغییر: باید 2x Rank باشه
    LORA_DROPOUT = 0.05  # ✅ تغییر: کمی dropout اضافه کردیم
    TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
    LORA_BIAS = "none"
    USE_GRADIENT_CHECKPOINTING = "unsloth"
    USE_RSLORA = False
    LOFTQ_CONFIG = None

    # Training Hyperparameters - اصلاح‌شده ✅
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
    PER_DEVICE_EVAL_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 4  # effective batch = 8
    LEARNING_RATE = 5e-5  # ✅ تغییر اصلی: کاهش از 2e-4
    NUM_TRAIN_EPOCHS = 5  # ✅ تغییر: افزایش از 3
    MAX_STEPS = -1
    WARMUP_RATIO = 0.03  # ✅ تغییر: کاهش warmup
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 0.3  # ✅ تغییر: کاهش برای stability

    # Scheduler & Optimizer
    LR_SCHEDULER_TYPE = "cosine"
    OPTIM = "adamw_8bit"

    # Evaluation & Saving - اصلاح‌شده ✅
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 100  # ✅ تغییر: افزایش از 50
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 100  # ✅ تغییر: افزایش از 50
    SAVE_TOTAL_LIMIT = 3  # ✅ تغییر: 3 بهترین checkpoint
    LOGGING_STEPS = 25  # ✅ تغییر: logging بیشتر
    LOAD_BEST_MODEL_AT_END = True
    METRIC_FOR_BEST_MODEL = "eval_loss"
    GREATER_IS_BETTER = False

    # Early Stopping - اصلاح‌شده ✅
    EARLY_STOPPING_PATIENCE = 5  # ✅ تغییر: افزایش از 3
    EARLY_STOPPING_THRESHOLD = 0.0001  # ✅ تغییر: دقت بیشتر

    # Memory Optimization
    BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
    FP16 = not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

    # Data Settings
    DATALOADER_NUM_WORKERS = 2
    DATALOADER_PIN_MEMORY = True
    GROUP_BY_LENGTH = True
    PACKING = False

    # Other
    SEED = 3407
    REPORT_TO = "none"

config = Config()

# نمایش تغییرات
print("="*80)
print("🔧 تنظیمات اصلاح‌شده:")
print("="*80)
print(f"✅ Learning Rate: {config.LEARNING_RATE} (قبلاً: 2e-4)")
print(f"✅ LoRA Alpha: {config.LORA_ALPHA} (قبلاً: 16)")
print(f"✅ LoRA Dropout: {config.LORA_DROPOUT} (قبلاً: 0)")
print(f"✅ Epochs: {config.NUM_TRAIN_EPOCHS} (قبلاً: 3)")
print(f"✅ Max Grad Norm: {config.MAX_GRAD_NORM} (قبلاً: 1.0)")
print(f"✅ Eval Steps: {config.EVAL_STEPS} (قبلاً: 50)")
print(f"✅ Early Stop Patience: {config.EARLY_STOPPING_PATIENCE} (قبلاً: 3)")
print(f"✅ Warmup Ratio: {config.WARMUP_RATIO} (قبلاً: 0.1)")
print("="*80)

🔧 تنظیمات اصلاح‌شده:
✅ Learning Rate: 5e-05 (قبلاً: 2e-4)
✅ LoRA Alpha: 32 (قبلاً: 16)
✅ LoRA Dropout: 0.05 (قبلاً: 0)
✅ Epochs: 5 (قبلاً: 3)
✅ Max Grad Norm: 0.3 (قبلاً: 1.0)
✅ Eval Steps: 100 (قبلاً: 50)
✅ Early Stop Patience: 5 (قبلاً: 3)
✅ Warmup Ratio: 0.03 (قبلاً: 0.1)


## بخش 3: Setup محیط

In [4]:
def setup_environment():
    """راه‌اندازی محیط"""
    print("\n" + "="*80)
    print("🔧 راه‌اندازی محیط")
    print("="*80)

    # Mount Drive
    try:
        print("📂 اتصال به Google Drive...")
        drive.mount(config.DRIVE_MOUNT)
        print("✅ Drive متصل شد")
    except Exception as e:
        print(f"⚠️ هشدار: {e}")

    # بررسی GPU
    if torch.cuda.is_available():
        gpu_stats = torch.cuda.get_device_properties(0)
        print(f"🎮 GPU: {gpu_stats.name}")
        print(f"💾 VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")

        vram_gb = gpu_stats.total_memory / 1024**3
        if vram_gb < 14:
            print("⚠️ هشدار: VRAM کم است. استفاده از Colab Pro توصیه می‌شود")
    else:
        raise RuntimeError("❌ GPU یافت نشد!")

    # ایجاد دایرکتوری‌ها
    Path(config.OUTPUT_DIR).mkdir(exist_ok=True)
    Path(config.FINAL_MODEL_DIR).mkdir(exist_ok=True)
    print("✅ محیط آماده است")

setup_environment()


🔧 راه‌اندازی محیط
📂 اتصال به Google Drive...
Mounted at /content/drive
✅ Drive متصل شد
🎮 GPU: Tesla T4
💾 VRAM: 14.74 GB
✅ محیط آماده است


## بخش 4: پردازش Dataset (HumanEval + MBPP)

In [5]:
class CodeDataProcessor:
    """پردازشگر dataset های کد"""

    @staticmethod
    def format_humaneval(example: Dict) -> Dict[str, str]:
        """فرمت HumanEval"""
        prompt = example.get('prompt', '').strip()
        canonical_solution = example.get('canonical_solution', '').strip()

        text = f"""<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Complete the following Python function:
{prompt}<|im_end|>
<|im_start|>assistant
{canonical_solution}<|im_end|>"""

        return {"text": text}

    @staticmethod
    def format_mbpp(example: Dict) -> Dict[str, str]:
        """فرمت MBPP"""
        text_description = (
            example.get('text', '') or
            example.get('prompt', '') or
            example.get('description', '')
        ).strip()

        code = (
            example.get('code', '') or
            example.get('canonical_solution', '') or
            example.get('solution', '')
        ).strip()

        text = f"""<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Write a Python function that: {text_description}<|im_end|>
<|im_start|>assistant
{code}<|im_end|>"""

        return {"text": text}

def load_and_process_datasets():
    """بارگذاری و ترکیب dataset ها"""
    print("\n" + "="*80)
    print("📚 بارگذاری Datasets")
    print("="*80)

    processor = CodeDataProcessor()

    # بارگذاری MBPP (اصلی)
    print("\n📥 بارگذاری MBPP...")
    try:
        mbpp = load_dataset("google-research-datasets/mbpp", "sanitized")

        mbpp_train = mbpp['train'].map(
            processor.format_mbpp,
            remove_columns=mbpp['train'].column_names,
            desc="Formatting MBPP Train"
        )

        mbpp_test = mbpp['test'].map(
            processor.format_mbpp,
            remove_columns=mbpp['test'].column_names,
            desc="Formatting MBPP Test"
        )

        print(f"   ✅ MBPP Train: {len(mbpp_train)} samples")
        print(f"   ✅ MBPP Test: {len(mbpp_test)} samples")

    except Exception as e:
        print(f"   ❌ خطا در بارگذاری MBPP: {e}")
        raise

    # بارگذاری HumanEval (اختیاری)
    print("\n📥 تلاش برای بارگذاری HumanEval...")
    humaneval_train = None

    try:
        humaneval = load_dataset("evalplus/humanevalplus")
        humaneval_train = humaneval['test'].map(
            processor.format_humaneval,
            remove_columns=humaneval['test'].column_names,
            desc="Formatting HumanEval"
        )
        print(f"   ✅ HumanEval: {len(humaneval_train)} samples")
    except Exception as e:
        print(f"   ⚠️ HumanEval بارگذاری نشد: {str(e)[:100]}")
        print("   ⏭️ ادامه با MBPP...")

    # ترکیب datasets
    if humaneval_train is not None:
        train_dataset = concatenate_datasets([humaneval_train, mbpp_train])
        print(f"\n✅ ترکیب: HumanEval + MBPP = {len(train_dataset)} samples")
    else:
        train_dataset = mbpp_train
        print(f"\n✅ Train dataset: {len(train_dataset)} samples")

    eval_dataset = mbpp_test

    print(f"\n📊 نتیجه نهایی:")
    print(f"   Total Train: {len(train_dataset)} samples")
    print(f"   Eval: {len(eval_dataset)} samples")

    # نمایش نمونه
    print("\n📋 نمونه فرمت‌شده:")
    print("-" * 80)
    print(train_dataset[0]['text'][:500] + "...")
    print("-" * 80)

    return train_dataset, eval_dataset

# بارگذاری datasets
train_dataset, eval_dataset = load_and_process_datasets()


📚 بارگذاری Datasets

📥 بارگذاری MBPP...


README.md: 0.00B [00:00, ?B/s]

sanitized/train-00000-of-00001.parquet:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

sanitized/test-00000-of-00001.parquet:   0%|          | 0.00/60.9k [00:00<?, ?B/s]

sanitized/validation-00000-of-00001.parq(…):   0%|          | 0.00/14.0k [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/551 [00:00<?, ?B/s]

data/test-00000-of-00001-5973903632b82d4(…):   0%|          | 0.00/2.90M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Formatting HumanEval:   0%|          | 0/164 [00:00<?, ? examples/s]

   ✅ HumanEval: 164 samples

✅ ترکیب: HumanEval + MBPP = 284 samples

📊 نتیجه نهایی:
   Total Train: 284 samples
   Eval: 257 samples

📋 نمونه فرمت‌شده:
--------------------------------------------------------------------------------
<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Complete the following Python function:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    ...
--------------------------------------------------------------------------------


## بخش 5: بارگذاری مدل با LoRA

In [6]:
def load_model_with_lora():
    """بارگذاری مدل با Unsloth"""

    print("\n" + "="*80)
    print("🚀 بارگذاری مدل با Unsloth")
    print("="*80)

    print(f"📥 بارگذاری {config.MODEL_NAME}...")

    # بارگذاری مدل و tokenizer
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.MODEL_NAME,
        max_seq_length=config.MAX_SEQ_LENGTH,
        dtype=config.DTYPE,
        load_in_4bit=config.LOAD_IN_4BIT,
    )

    print("✅ مدل بارگذاری شد")

    # اضافه کردن LoRA adapters
    print("\n🔧 اضافه کردن LoRA adapters...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.LORA_R,
        lora_alpha=config.LORA_ALPHA,
        lora_dropout=config.LORA_DROPOUT,
        target_modules=config.TARGET_MODULES,
        bias=config.LORA_BIAS,
        use_gradient_checkpointing=config.USE_GRADIENT_CHECKPOINTING,
        random_state=config.SEED,
        use_rslora=config.USE_RSLORA,
        loftq_config=config.LOFTQ_CONFIG,
    )

    # محاسبه آمار
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percent = 100 * trainable_params / total_params

    print(f"✅ LoRA adapters اضافه شد")
    print(f"\n📊 آمار پارامترها:")
    print(f"   • کل: {total_params:,}")
    print(f"   • قابل آموزش: {trainable_params:,} ({trainable_percent:.3f}%)")
    print(f"\n⚙️ تنظیمات LoRA:")
    print(f"   • Rank: {config.LORA_R}")
    print(f"   • Alpha: {config.LORA_ALPHA}")
    print(f"   • Dropout: {config.LORA_DROPOUT}")

    return model, tokenizer

model, tokenizer = load_model_with_lora()

# نمایش وضعیت GPU
print("\n" + "="*80)
print("💾 وضعیت حافظه GPU")
print("="*80)
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"استفاده شده: {used_memory} GB / {total_memory} GB")
print("="*80)


🚀 بارگذاری مدل با Unsloth
📥 بارگذاری unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit...
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ مدل بارگذاری شد

🔧 اضافه کردن LoRA adapters...


Unsloth 2025.12.5 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA adapters اضافه شد

📊 آمار پارامترها:
   • کل: 8,232,817,664
   • قابل آموزش: 68,812,800 (0.836%)

⚙️ تنظیمات LoRA:
   • Rank: 16
   • Alpha: 32
   • Dropout: 0.05

💾 وضعیت حافظه GPU
GPU: Tesla T4
استفاده شده: 9.56 GB / 14.74 GB


## بخش 6: Training

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

def train_model():
    """آموزش مدل"""
    print("\n" + "="*80)
    print("🚀 شروع Fine-tuning")
    print("="*80)

    # تنظیمات training با SFTConfig
    training_args = SFTConfig(
        output_dir=config.OUTPUT_DIR,

        # Batch
        per_device_train_batch_size=config.PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=config.PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,

        # Optimization
        learning_rate=config.LEARNING_RATE,
        warmup_ratio=config.WARMUP_RATIO,
        lr_scheduler_type=config.LR_SCHEDULER_TYPE,
        weight_decay=config.WEIGHT_DECAY,
        optim=config.OPTIM,
        max_grad_norm=config.MAX_GRAD_NORM,

        # Epochs
        num_train_epochs=config.NUM_TRAIN_EPOCHS,
        max_steps=config.MAX_STEPS,

        # Precision
        fp16=config.FP16,
        bf16=config.BF16,

        # Logging & Saving
        logging_steps=config.LOGGING_STEPS,
        save_steps=config.SAVE_STEPS,
        save_total_limit=config.SAVE_TOTAL_LIMIT,
        eval_steps=config.EVAL_STEPS,

        # Evaluation
        eval_strategy=config.EVAL_STRATEGY,
        load_best_model_at_end=config.LOAD_BEST_MODEL_AT_END,
        metric_for_best_model=config.METRIC_FOR_BEST_MODEL,
        greater_is_better=config.GREATER_IS_BETTER,

        # Other
        report_to=config.REPORT_TO,
        seed=config.SEED,
        dataloader_num_workers=config.DATALOADER_NUM_WORKERS,
        group_by_length=config.GROUP_BY_LENGTH,

        # SFT specific
        dataset_text_field="text",
        max_seq_length=config.MAX_SEQ_LENGTH,
        packing=config.PACKING,
    )

    effective_batch = config.PER_DEVICE_TRAIN_BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS
    total_steps = (len(train_dataset) * config.NUM_TRAIN_EPOCHS) // effective_batch

    print(f"📊 تنظیمات Training:")
    print(f"   • Train samples: {len(train_dataset)}")
    print(f"   • Eval samples: {len(eval_dataset)}")
    print(f"   • Batch size: {config.PER_DEVICE_TRAIN_BATCH_SIZE}")
    print(f"   • Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"   • Effective batch: {effective_batch}")
    print(f"   • Epochs: {config.NUM_TRAIN_EPOCHS}")
    print(f"   • Total steps: ~{total_steps}")
    print(f"   • Learning rate: {config.LEARNING_RATE}")
    print("-" * 80)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config.EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=config.EARLY_STOPPING_THRESHOLD
            )
        ]
    )

    print("\n🔥 Training شروع شد...\n")
    trainer.train()
    print("\n✅ Training با موفقیت تکمیل شد!")

    return trainer

# شروع training
trainer = train_model()


🚀 شروع Fine-tuning
📊 تنظیمات Training:
   • Train samples: 284
   • Eval samples: 257
   • Batch size: 2
   • Gradient accumulation: 4
   • Effective batch: 8
   • Epochs: 5
   • Total steps: ~177
   • Learning rate: 5e-05
--------------------------------------------------------------------------------


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/284 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/257 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.

🔥 Training شروع شد...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 284 | Num Epochs = 5 | Total steps = 180
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
100,0.398200,0.523296


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ Training با موفقیت تکمیل شد!


## بخش 7: ذخیره مدل

In [8]:
def save_model():
    """ذخیره مدل نهایی"""
    print("\n" + "="*80)
    print("💾 ذخیره مدل")
    print("="*80)

    # ذخیره local
    print(f"📁 ذخیره در {config.FINAL_MODEL_DIR}...")
    model.save_pretrained(config.FINAL_MODEL_DIR)
    tokenizer.save_pretrained(config.FINAL_MODEL_DIR)
    print("✅ ذخیره محلی انجام شد")

    # ذخیره در Drive
    try:
        print(f"\n📁 ذخیره در Google Drive...")
        import shutil
        if Path(config.DRIVE_SAVE_DIR).exists():
            shutil.rmtree(config.DRIVE_SAVE_DIR)
        shutil.copytree(config.FINAL_MODEL_DIR, config.DRIVE_SAVE_DIR)
        print(f"✅ ذخیره در Drive: {config.DRIVE_SAVE_DIR}")
    except Exception as e:
        print(f"⚠️ خطا در ذخیره Drive: {e}")

    print("\n✅ مدل با موفقیت ذخیره شد!")
    print("="*80)

save_model()


💾 ذخیره مدل
📁 ذخیره در ./qwen-coder-final...
✅ ذخیره محلی انجام شد

📁 ذخیره در Google Drive...
✅ ذخیره در Drive: /content/drive/MyDrive/qwen-coder-humaneval

✅ مدل با موفقیت ذخیره شد!


## بخش 8: تست مدل

In [9]:
def test_model():
    """تست سریع مدل"""
    print("\n" + "="*80)
    print("🧪 تست مدل")
    print("="*80)

    # فعال‌سازی حالت inference
    FastLanguageModel.for_inference(model)

    test_prompts = [
        # 1️⃣ پایه‌ای (MBPP)
        "Write a Python function that returns the sum of all even numbers in a list.",

        # 2️⃣ رشته‌ای
        "Write a Python function that checks whether a string is a palindrome, ignoring spaces and case.",

        # 3️⃣ منطقی کوتاه
        "Write a Python function that returns the first non-repeating character in a string.",

        # 4️⃣ الگوریتمی سبک (HumanEval-like)
        "Write a Python function that merges two sorted lists into one sorted list.",

        # 5️⃣ لبه‌ای / استدلالی
        "Write a Python function that returns True if a number is a prime number, otherwise False."
    ]

    for i, problem in enumerate(test_prompts, 1):
        print(f"\n{'='*80}")
        print(f"🧪 تست {i}")
        print(f"{'='*80}")

        test_prompt = f"""<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
{problem}<|im_end|>
<|im_start|>assistant
"""

        print("📝 پرامپت تست:")
        print(test_prompt)
        print("\n🤖 پاسخ مدل:")
        print("-" * 80)

        inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        print(response[len(test_prompt):])
        print("-" * 80)

    print("\n✅ همه تست‌ها کامل شد!")


test_model()

print("\n" + "="*80)
print("🎉 پروژه با موفقیت تکمیل شد!")
print("="*80)



🧪 تست مدل

🧪 تست 1
📝 پرامپت تست:
<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Write a Python function that returns the sum of all even numbers in a list.<|im_end|>
<|im_start|>assistant


🤖 پاسخ مدل:
--------------------------------------------------------------------------------
def sum_of_evens(numbers):
    return sum(x for x in numbers if x % 2 == 0)<|im_end|>
--------------------------------------------------------------------------------

🧪 تست 2
📝 پرامپت تست:
<|im_start|>system
You are an expert Python programmer. Write clean, efficient, and correct code.<|im_end|>
<|im_start|>user
Write a Python function that checks whether a string is a palindrome, ignoring spaces and case.<|im_end|>
<|im_start|>assistant


🤖 پاسخ مدل:
--------------------------------------------------------------------------------
def is_palindrome(s: str) -> bool:
    s = ''.join([x for x in s if x != ' ']).lower()
    return s 